# ⚡ RevertIQ — Full Workflow

**Cross-Sectional Mean Reversion Swing Trading for Indian Equity Markets**

This notebook runs the complete pipeline end-to-end:

1. 📥 **Data Download** — NIFTY 50 OHLCV from yfinance
2. 🧹 **Data Cleaning** — Handle gaps, validate integrity
3. 📊 **Technical Indicators** — RSI, Z-score, ATR, BB, etc.
4. 🧮 **Feature Engineering** — Cross-sectional features
5. 🚦 **Regime Filter** — Only trade in bull markets
6. 📡 **Signal Generation** — Multi-condition BUY/SELL
7. 🏆 **Stock Ranking** — Weighted composite scoring
8. 🔬 **Backtesting** — Realistic portfolio simulation
9. 📈 **Visualization** — Charts and reports
10. 📋 **Daily Watchlist** — Actionable trade candidates

---
## 0. Setup & Installation

In [1]:
# ── Google Colab Setup ──
# Uncomment these lines if running in Google Colab:

# !git clone https://github.com/elvishpatel/RevertIQ.git /content/RevertIQ
# %cd /content/RevertIQ
# !pip install -e . -q

# ── Local Setup ──
# If running locally, make sure you've installed: pip install -e .

import warnings
warnings.filterwarnings('ignore')

# Verify installation
import revertiq
print(f"RevertIQ v{revertiq.__version__} loaded successfully ✓")

ModuleNotFoundError: No module named 'revertiq'

In [2]:
# ── Import all modules ──
from revertiq.config.settings import get_default_config, RevertIQConfig
from revertiq.data.universe import get_nifty50_tickers, get_index_tickers
from revertiq.data.downloader import DataDownloader
from revertiq.data.cleaner import DataCleaner
from revertiq.indicators.technical import TechnicalIndicators
from revertiq.features.engineering import FeatureEngineer
from revertiq.signals.regime import RegimeFilter
from revertiq.signals.generator import SignalGenerator
from revertiq.ranking.ranker import StockRanker
from revertiq.backtesting.engine import BacktestEngine
from revertiq.backtesting.metrics import PerformanceMetrics
from revertiq.visualization.charts import ChartEngine
from revertiq.visualization.reports import ReportGenerator

import pandas as pd
import numpy as np

print("All modules imported ✓")

ModuleNotFoundError: No module named 'revertiq'

In [ ]:
# ── Configuration ──
config = get_default_config()

# Customize if needed:
# from dataclasses import replace
# config.signal = replace(config.signal, rsi_buy_threshold=15)
# config.backtest = replace(config.backtest, max_positions=10)

print(f"Initial capital  : ₹{config.backtest.initial_capital:,.0f}")
print(f"Max positions    : {config.backtest.max_positions}")
print(f"RSI threshold    : {config.signal.rsi_buy_threshold}")
print(f"Z-score threshold: {config.signal.zscore_buy_threshold}")
print(f"Date range       : {config.data.start_date} → {config.data.end_date}")

---
## 1. 📥 Data Download

In [ ]:
# Download NIFTY 50 stock data
downloader = DataDownloader(config.data)

# Show universe
tickers = get_nifty50_tickers()
print(f"Universe: {len(tickers)} NIFTY 50 stocks")
print(f"Sample: {tickers[:5]}")

In [ ]:
# Download all data (takes 2-5 minutes)
raw_data = downloader.download_all()
print(f"\nDownloaded {len(raw_data)} tickers ✓")

In [ ]:
# Download index data (NIFTY 50 + India VIX)
index_data = downloader.download_index_data()
for name, df in index_data.items():
    print(f"{name}: {len(df)} rows, {df.index[0].date()} → {df.index[-1].date()}")

---
## 2. 🧹 Data Cleaning

In [ ]:
# Clean and validate all data
cleaner = DataCleaner(config.data)
clean_data = cleaner.clean_all()

# Show cleaning report
report = cleaner.get_cleaning_report()
print(f"\nCleaning Summary:")
print(f"  Tickers processed : {report.get('tickers_processed', 'N/A')}")
print(f"  Tickers kept      : {report.get('tickers_kept', 'N/A')}")
print(f"  Tickers dropped   : {report.get('tickers_dropped', 'N/A')}")

In [ ]:
# Combine into a single DataFrame for cross-sectional analysis
combined_df = cleaner.get_combined_df()
print(f"Combined DataFrame: {combined_df.shape}")
print(f"Tickers: {combined_df['ticker'].nunique()}")
print(f"Date range: {combined_df['date'].min().date()} → {combined_df['date'].max().date()}")
combined_df.head()

---
## 3. 📊 Technical Indicators

In [ ]:
# Compute indicators for each stock
ti = TechnicalIndicators()
enriched_dfs = []

for ticker in combined_df['ticker'].unique():
    stock_df = combined_df[combined_df['ticker'] == ticker].copy()
    stock_df = stock_df.sort_values('date')
    enriched = ti.compute_all(stock_df, config.indicator)
    enriched_dfs.append(enriched)

stock_data = pd.concat(enriched_dfs, ignore_index=True)
print(f"Indicators computed for {stock_data['ticker'].nunique()} stocks ✓")
print(f"Columns: {list(stock_data.columns)}")

---
## 4. 🧮 Cross-Sectional Features

In [ ]:
# Prepare market (NIFTY) data
nifty_data = index_data.get('nifty50', pd.DataFrame())
if not nifty_data.empty:
    nifty_data.columns = [c.lower() for c in nifty_data.columns]
    if 'date' not in nifty_data.columns:
        nifty_data = nifty_data.reset_index()
        nifty_data.columns = [c.lower() for c in nifty_data.columns]

# Compute cross-sectional features
fe = FeatureEngineer(config.indicator, config.ranking)
features_df = fe.compute_all_features(stock_data, nifty_data)
print(f"Features computed: {features_df.shape}")
print(f"New columns: {[c for c in features_df.columns if c not in stock_data.columns]}")

---
## 5. 🚦 Market Regime Filter

In [ ]:
# Compute market regime
rf = RegimeFilter(config.regime)

vix_data = index_data.get('india_vix', None)
if vix_data is not None and not vix_data.empty:
    vix_data.columns = [c.lower() for c in vix_data.columns]

# Need NIFTY with proper index
nifty_for_regime = nifty_data.copy()
if 'date' in nifty_for_regime.columns:
    nifty_for_regime = nifty_for_regime.set_index('date')

regime_series = rf.compute_regime(nifty_for_regime, vix_data)

# Show regime distribution
regime_counts = regime_series.value_counts()
print("Market Regime Distribution:")
for regime, count in regime_counts.items():
    pct = count / len(regime_series) * 100
    print(f"  {regime:15s}: {count:5d} days ({pct:.1f}%)")

In [ ]:
# Visualize regime
ce = ChartEngine()
fig = ce.regime_chart(nifty_for_regime.reset_index(), regime_series)
fig.show()

---
## 6. 📡 Signal Generation

In [ ]:
# Generate BUY and SELL signals
sg = SignalGenerator(config.signal)
signals_df = sg.generate_all_signals(features_df, regime_series)

n_buy = signals_df['buy_signal'].sum()
n_sell = signals_df['sell_signal'].sum()
print(f"BUY signals : {n_buy:,d}")
print(f"SELL signals: {n_sell:,d}")
print(f"Buy signal rate: {n_buy/len(signals_df)*100:.2f}%")

In [ ]:
# Show signal distribution over time
buy_by_date = signals_df[signals_df['buy_signal']].groupby('date').size()
import plotly.express as px
fig = px.bar(x=buy_by_date.index, y=buy_by_date.values,
             title='Daily BUY Signal Count',
             labels={'x': 'Date', 'y': 'Number of Signals'})
fig.update_layout(template='plotly_dark',
                  paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
fig.show()

---
## 7. 🏆 Stock Ranking

In [ ]:
# Rank stocks on each day with buy signals
ranker = StockRanker(config.ranking)
rankings_df = ranker.rank_all_dates(features_df, signals_df)

print(f"Ranked candidates: {len(rankings_df):,d} across {rankings_df['date'].nunique()} dates")
print(f"\nLatest top candidates:")
top = ranker.get_top_candidates(rankings_df, n=10)
if not top.empty:
    display_cols = [c for c in ['rank', 'ticker', 'composite_score', 'signal_strength'] if c in top.columns]
    print(top[display_cols].to_string(index=False))

In [ ]:
# Visualize ranked stocks table
fig = ce.ranked_stocks_table(rankings_df)
fig.show()

---
## 8. 🔬 Backtesting

In [ ]:
# Prepare data for backtester
# Ensure 'date' column exists (not just index)
backtest_data = features_df.copy()
if 'date' not in backtest_data.columns:
    backtest_data = backtest_data.reset_index()

# Prepare benchmark
benchmark = nifty_data.copy()
if 'date' not in benchmark.columns:
    benchmark = benchmark.reset_index()

print(f"Backtest data: {backtest_data.shape}")
print(f"Signals: {signals_df.shape}")
print(f"Rankings: {rankings_df.shape}")

In [ ]:
# Run the backtest!
engine = BacktestEngine(config)
result = engine.run_with_benchmark(
    price_data=backtest_data,
    signals_df=signals_df,
    rankings_df=rankings_df,
    regime_series=regime_series,
    benchmark_data=benchmark,
)

# Print performance summary
rg = ReportGenerator()
rg.print_metrics_summary(result.metrics)

---
## 9. 📈 Visualization

In [ ]:
# 9.1 — Equity Curve
fig = ce.equity_curve(result.equity_curve, result.benchmark_equity)
fig.show()

In [ ]:
# 9.2 — Drawdown
if not result.daily_returns.empty:
    dd = PerformanceMetrics.drawdown_series(result.equity_curve)
    if dd is not None and not dd.empty:
        fig = ce.drawdown_chart(dd)
        fig.show()

In [ ]:
# 9.3 — Rolling Sharpe
if not result.daily_returns.empty:
    rs = PerformanceMetrics.rolling_sharpe(result.daily_returns)
    if rs is not None and not rs.empty:
        fig = ce.rolling_sharpe_chart(rs)
        fig.show()

In [ ]:
# 9.4 — Monthly Returns Heatmap
if not result.daily_returns.empty:
    monthly = PerformanceMetrics.monthly_returns(result.daily_returns)
    if monthly is not None and not monthly.empty:
        fig = ce.monthly_returns_heatmap(monthly)
        fig.show()

In [ ]:
# 9.5 — Return Distribution
if not result.daily_returns.empty:
    fig = ce.return_distribution(result.daily_returns)
    fig.show()

In [ ]:
# 9.6 — Signal chart for a specific stock
sample_ticker = 'RELIANCE.NS'
if sample_ticker in features_df['ticker'].unique():
    stock_prices = features_df[features_df['ticker'] == sample_ticker].copy()
    stock_signals = signals_df[signals_df['ticker'] == sample_ticker].copy()
    
    if 'date' in stock_prices.columns:
        stock_prices = stock_prices.set_index('date')
    if 'date' in stock_signals.columns:
        stock_signals = stock_signals.set_index('date')
    
    buys = stock_signals['buy_signal'] if 'buy_signal' in stock_signals.columns else None
    sells = stock_signals['sell_signal'] if 'sell_signal' in stock_signals.columns else None
    
    fig = ce.signal_chart(sample_ticker, stock_prices, buys, sells)
    fig.show()

---
## 10. 📋 Daily Watchlist & Exports

In [ ]:
# Generate today's watchlist
print("\n" + "=" * 60)
print("  ⚡ RevertIQ — DAILY WATCHLIST")
print("=" * 60)

latest_date = rankings_df['date'].max() if not rankings_df.empty else 'N/A'
print(f"  Date: {latest_date}")
print()

top_5 = ranker.get_top_candidates(rankings_df, n=5)
if not top_5.empty:
    display_cols = [c for c in ['rank', 'ticker', 'composite_score', 'signal_strength'] if c in top_5.columns]
    print(top_5[display_cols].to_string(index=False))
else:
    print("  No candidates today.")
print("=" * 60)

In [ ]:
# Export reports
rg = ReportGenerator()

# HTML backtest report
report_path = rg.generate_backtest_report(result, output_dir='data/results')
print(f"HTML Report → {report_path}")

# CSV exports
if not result.trade_log.empty:
    rg.export_trade_log(result.trade_log)

if not result.equity_curve.empty:
    rg.export_equity_curve(result.equity_curve)

if not rankings_df.empty:
    rg.export_daily_watchlist(rankings_df)

print("\nAll exports complete ✓")

---
## 📊 Trade Analysis

In [ ]:
# Analyze completed trades
if not result.trade_log.empty:
    tl = result.trade_log
    print(f"Total trades: {len(tl)}")
    print(f"Winning trades: {(tl['pnl'] > 0).sum()} ({(tl['pnl'] > 0).mean()*100:.1f}%)")
    print(f"Losing trades: {(tl['pnl'] <= 0).sum()} ({(tl['pnl'] <= 0).mean()*100:.1f}%)")
    print(f"\nAvg PnL: ₹{tl['pnl'].mean():,.0f}")
    print(f"Avg win: ₹{tl[tl['pnl']>0]['pnl'].mean():,.0f}")
    print(f"Avg loss: ₹{tl[tl['pnl']<=0]['pnl'].mean():,.0f}")
    print(f"\nAvg holding: {tl['holding_days'].mean():.1f} days")
    print(f"\nExit reasons:")
    print(tl['exit_reason'].value_counts().to_string())
else:
    print("No trades were executed. Check signal thresholds and regime filter.")

---
## 🎯 Next Steps

1. **Tune parameters**: Adjust RSI threshold, Z-score threshold, max holding period
2. **Walk-forward optimization**: Use `WalkForwardOptimizer` to find robust parameters
3. **Expand universe**: Try NIFTY 100 or NIFTY 200
4. **Add ML ranking**: Use Random Forest / XGBoost to predict reversion probability
5. **Build dashboard**: Create a Streamlit app for daily monitoring

In [3]:
import revertiq

ModuleNotFoundError: No module named 'revertiq'